<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session01_getting_started_with_llms_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱: شروع کار با مدل‌های زبانی بزرگ

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session01_getting_started_with_llms.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import time


def measure(prompt, model, max_tokens=300):
    """Return the answer, the elapsed time and the generation speed in tokens per second."""
    start = time.time()
    response = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], max_tokens=max_tokens, temperature=0)
    elapsed = time.time() - start
    tokens = response.usage.completion_tokens
    return response.choices[0].message.content, elapsed, tokens / elapsed

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

مدل `gemma3:4b` یا `llama3.2:3b` را با `setup_llm("ollama", model="...")` اجرا کنید. پاسخ آن به سه پرسش فارسی را با `qwen2.5:7b` مقایسه کنید و سرعت هر دو را با تابع `measure` بسنجید.

</div>

In [ ]:
QUESTIONS = [
    "در دو جمله توضیح بده یادگیری ماشین چیست.",
    "سه پیشنهاد برای کاهش مصرف برق در خانه بده.",
    "جمله «I will call you tomorrow» را به فارسی محاوره‌ای ترجمه کن.",
]

for model in ["qwen2.5:7b", "gemma3:4b"]:
    client, MODEL = setup_llm("ollama", model=model)
    print(f"===== {model}")
    for q in QUESTIONS:
        answer, seconds, speed = measure(q, MODEL)
        print(f"\nQ: {q}\n({seconds:.1f}s, {speed:.0f} tok/s)\n{answer}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`setup_llm` هر مدل را فقط یک بار دانلود می‌کند و Ollama چند مدل را کنار هم نگه می‌دارد. هنگام مقایسه به این نکات توجه کنید:

- **سرعت:** مدل ۴ میلیارد پارامتری معمولاً سریع‌تر از مدل ۷ میلیاردی است، چون در هر گام محاسبهٔ کمتری لازم دارد.
- **کیفیت فارسی:** مدل‌های کوچک‌تر بیشتر دچار خطای دستوری، جملهٔ نیمه‌انگلیسی یا ترجمهٔ تحت‌اللفظی می‌شوند. ترجمهٔ محاوره‌ای سخت‌ترین پرسش این مجموعه است.
- **منصفانه بودن مقایسه:** سه پرسش برای نتیجه‌گیری کافی نیست. روش درست مقایسه را در جلسهٔ ۵ می‌بینیم. اجرای اول هر مدل هم کندتر است، چون مدل باید در حافظهٔ GPU بارگذاری شود؛ اجرای اول را در زمان‌سنجی حساب نکنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

یک پاراگراف فارسی و ترجمه انگلیسی آن را انتخاب کنید و تعداد توکن‌هایشان را با توکن‌ساز Qwen بشمارید. نسبت توکن‌ها چقدر است؟ این نسبت چه اثری روی هزینه یک چت‌بات فارسی دارد؟

</div>

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

persian = ("نوین‌کالا سفارش‌ها را به همه شهرهای ایران ارسال می‌کند. ارسال عادی سه تا پنج روز کاری "
           "طول می‌کشد و برای خریدهای بیش از یک میلیون تومان رایگان است.")
english = ("Novin Kala ships orders to every city in Iran. Standard shipping takes three to five "
           "working days and is free for purchases over one million toman.")

n_fa, n_en = len(tokenizer(persian).input_ids), len(tokenizer(english).input_ids)
print(f"Persian: {len(persian)} characters, {n_fa} tokens")
print(f"English: {len(english)} characters, {n_en} tokens")
print(f"ratio Persian / English: {n_fa / n_en:.2f}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نسبت دقیق به متن بستگی دارد، ولی برای توکن‌سازهای چندزبانهٔ امروزی معمولاً متن فارسی بیشتر از معادل انگلیسی‌اش توکن مصرف می‌کند.

**اثر روی هزینه:** اگر نسبت $r$ باشد، یک چت‌بات فارسی با همان تعداد گفتگو تقریباً $r$ برابر هزینهٔ توکن دارد. مثلاً اگر چت‌بات انگلیسی ماهانه ۱۰۰ دلار هزینه داشته باشد و $r = 1.8$ باشد، نسخهٔ فارسی حدود ۱۸۰ دلار هزینه دارد. همین نسبت روی سرعت تولید پاسخ و ظرفیت پنجرهٔ متن هم اثر دارد. یکی از دلایل اهمیت مدل‌هایی با توکن‌ساز بهتر برای فارسی همین است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

تابع `ask` را طوری تغییر دهید که فقط ۶ پیام آخر تاریخچه، به علاوه پیام system، ارسال شود. گفتگویی طراحی کنید که نشان دهد مدل چه چیزی را فراموش می‌کند.

</div>

In [ ]:
SYSTEM = {"role": "system", "content": "تو یک دستیار فارسی‌زبان هستی و خیلی کوتاه پاسخ می‌دهی."}
messages = []


def ask(question, window=6):
    """Send only the system message and the last `window` messages."""
    messages.append({"role": "user", "content": question})
    answer = chat(client, MODEL, messages=[SYSTEM] + messages[-window:], temperature=0)
    messages.append({"role": "assistant", "content": answer})
    return answer


conversation = [
    "اسم من کیوان است و شهرم آمل است.",
    "پایتخت فرانسه کجاست؟",
    "دو به علاوه سه چند می‌شود؟",
    "یک رنگ گرم نام ببر.",
    "اسم من چه بود و کجا زندگی می‌کنم؟",
]
for q in conversation:
    print(f"Q: {q}\nA: {ask(q)}\n")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

وقتی پرسش آخر ارسال می‌شود، پنجرهٔ ۶ پیامی شامل پیام اول کاربر (نام و شهر) نیست: پیام‌های پرسش ۲ تا ۴ و پاسخ‌هایشان شش پیام‌اند و پرسش پنجم هم اضافه می‌شود. پس مدل نام و شهر را نمی‌داند و یا می‌گوید نمی‌داند یا حدس می‌زند. اگر پنجره را به ۱۰ پیام برسانید، جواب درست می‌شود.

پنجرهٔ ثابت ساده‌ترین روش مدیریت حافظه است، ولی اطلاعات مهم اول گفتگو را دور می‌ریزد. در جلسهٔ ۱۵ روش‌های بهتر، مثل خلاصه‌سازی و نگهداری اطلاعات کلیدی در یک «وضعیت» جدا، را می‌بینیم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

با `temperature=0.7` و یک `seed` ثابت، یک درخواست را سه بار اجرا کنید. آیا خروجی‌ها یکسان است؟ بدون `seed` چطور؟

</div>

In [ ]:
prompt = "یک جملهٔ کوتاه و خلاقانه دربارهٔ باران بنویس."

for seed in [42, None]:
    print(f"--- seed={seed}")
    for _ in range(3):
        kwargs = {"seed": seed} if seed is not None else {}
        print(chat(client, MODEL, prompt, temperature=0.7, max_tokens=60, **kwargs).strip())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با `seed` ثابت، Ollama روی یک سخت‌افزار معمولاً هر سه بار همان خروجی را تولید می‌کند، چون مولد اعداد تصادفی نمونه‌برداری از نقطهٔ یکسانی شروع می‌شود. بدون `seed` خروجی‌ها با هم فرق دارند.

دو نکته:

- در سرویس‌های API مثل OpenAI، `seed` فقط تلاشی برای تکرارپذیری است و تضمینی نیست، چون درخواست‌ها روی سخت‌افزارهای مختلف و در دسته‌های متفاوت پردازش می‌شوند و محاسبات ممیز شناور نتایج کمی متفاوت می‌دهند.
- برای کارهایی که باید تکرارپذیر باشند، مثل استخراج اطلاعات و ارزیابی، از `temperature=0` استفاده کنید و `seed` را فقط ابزار کمکی بدانید.

</div>